# Chấm điểm gian lận trên luồng (Colab, CPU)

Colab không chạy được Docker, nên notebook này thay Redpanda + Redis bằng vòng lặp phát lại và `MemoryWindows` (cùng
logic với `RedisWindows`, có test so khớp). Muốn chạy luồng thật với Grafana: dùng máy có Docker và làm theo README.

In [ ]:
REPO_URL = "https://github.com/tak2k5/risk-ml-platform.git"   # sửa sau khi đẩy repo lên GitHub
import os, sys
if not os.path.exists("risk-ml-platform"):
    !git clone -q {REPO_URL}
%cd risk-ml-platform/fraud
!pip install -q lightgbm tabulate
sys.path[:0] = ["src", "../riskops/src"]  # riskops: thư viện dùng chung của repo

## 1. Dữ liệu

PaySim cần Kaggle API: tải `kaggle.json` (Kaggle > Settings > Create New Token) rồi chạy ô dưới. Để `USE_PAYSIM = False`
thì dùng dữ liệu giả lập: chỉ để thử đường ống, **không phải kết quả**.

In [ ]:
USE_PAYSIM = True

from fraudrt.data import load_paysim, make_synthetic
if USE_PAYSIM:
    if not os.path.exists("data/paysim.csv"):
        from google.colab import files
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        up = files.upload()  # chọn kaggle.json
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "wb").write(up["kaggle.json"])
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        !pip install -q kaggle && bash scripts/download_data.sh
    df = load_paysim("data/paysim.csv")
else:
    df = make_synthetic()
print(f"{len(df):,} giao dịch, gian lận {df.is_fraud.mean():.3%}")
df.groupby("type").is_fraud.agg(["size", "sum"])

Gian lận PaySim chỉ nằm ở TRANSFER và CASH_OUT, và tỉ lệ rất nhỏ: vì vậy chỉ số chính là PR-AUC, mốc so sánh là tỉ lệ
gian lận (đoán ngẫu nhiên), không phải accuracy.

## 2. Huấn luyện (chia theo thời gian) và ablation đặc trưng cửa sổ

In [ ]:
from fraudrt.train import run
res = run(df, "models", "reports", "PaySim" if USE_PAYSIM else "GIẢ LẬP")
print(open("reports/train.md", encoding="utf-8").read())

## 3. Phát lại như luồng: chấm từng giao dịch

Bắt đầu ở 85% thời gian (phần test), kho cửa sổ rỗng giống consumer vừa khởi động. Đo thời gian tính đặc trưng + mô hình
cho từng giao dịch (không gồm mạng và hàng đợi như khi chạy với Kafka).

In [ ]:
import numpy as np
from fraudrt.features import MemoryWindows
from fraudrt.scorer import Scorer

stream = df.iloc[int(len(df) * 0.85):].head(20000)
scorer = Scorer(MemoryWindows(), "models")
out = [scorer.score(tx) for tx in stream.to_dict("records")]
ms = np.array([r["score_ms"] for r in out])
alert = np.array([r["alert"] for r in out])
fraud = stream.is_fraud.to_numpy().astype(bool)
print(f"{len(out)} giao dịch; thời gian chấm p50 {np.percentile(ms, 50):.2f} ms, p95 {np.percentile(ms, 95):.2f} ms, "
      f"p99 {np.percentile(ms, 99):.2f} ms")
print(f"cảnh báo {alert.sum()}, đúng {(alert & fraud).sum()}, gian lận trong đoạn {fraud.sum()}")

## 4. Kiểm tra lệch đặc trưng

Đặc trưng lúc chấm từng giao dịch phải trùng với đặc trưng tính lại offline (phát lại cùng đoạn dữ liệu).

In [ ]:
import pandas as pd
from fraudrt.features import FEATURES, offline_features
online = pd.DataFrame([r["features"] for r in out], index=stream.tx_id.to_numpy())
offline = offline_features(stream)
diff = (online[FEATURES] - offline.loc[online.index, FEATURES]).abs()
print("số giá trị lệch > 1e-6:", int((diff > 1e-6).sum().sum()))
diff.max().to_frame("lệch lớn nhất")

## 5. Giao dịch bị cảnh báo

In [ ]:
show = stream.assign(score=[r["score"] for r in out], alert=alert)
show[show.alert].sort_values("score", ascending=False).head(10)[
    ["tx_id", "type", "amount", "orig", "dest", "old_bal_orig", "new_bal_orig", "score", "is_fraud"]]

## 6. Lưu kết quả vào Drive

Chép `reports/` sang `MyDrive/fraud/` để không mất khi Colab ngắt.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/fraud && cp -r reports /content/drive/MyDrive/fraud/